## Costo de la deuda y tasa impositiva

El **costo de la deuda después de impuestos** es el tercer componente para armar el
costo de capital de Apple. Se calcula en tres pasos:

$$ k_{d,\text{d.i.}} = k_d \times (1 - T) $$

donde $k_d$ es el costo de la deuda **antes** de impuestos y $T$ la tasa impositiva.
El factor $(1 - T)$ recoge el **escudo fiscal**: los intereses son deducibles, así que
la deuda le cuesta a la empresa menos que su tasa nominal.

**¿Qué pasivos cuentan?** Siguiendo el criterio de los casos del Excel (documentos por
pagar, bonos, deuda a largo plazo), solo entran los pasivos que **generan intereses**.
Para Apple, del balance del **10-Q (28-mar-2026)** eso significa:

- **Commercial paper** (deuda de corto plazo)
- **Term debt** — porción corriente y no corriente (los *Notes* de Apple)

Se **excluyen** los pasivos operativos que no pagan interés: cuentas por pagar,
otros pasivos corrientes y no corrientes, e ingresos diferidos.

**Fuentes de los datos:**

- **Montos de deuda** y **tasa impositiva**: 10-Q de Apple (Q2 FY2026, 28-mar-2026).
- **Tasas de interés efectivas** de la deuda: 10-K de Apple (FY2025, 27-sep-2025),
  Nota 9 – *Debt*, que reporta la tasa efectiva de cada tramo (el 10-Q no la desglosa).

In [1]:
import pandas as pd

# === Montos de deuda con intereses — 10-Q, 28-mar-2026 (US$ millones) ===
cp        = 1997     # Commercial paper (pasivo corriente)
td_corr   = 8310     # Term debt, porcion corriente
td_nocorr = 74404    # Term debt, no corriente
term_debt = td_corr + td_nocorr
deuda_total = cp + term_debt          # D = deuda total con intereses

# === Tasas de interes efectivas — 10-K, 27-sep-2025 (Nota 9) ===
tasa_cp = 0.0419     # Commercial paper: tasa promedio ponderada = 4.19%

# Term debt: la tasa efectiva se reporta por tramos, en rangos. Usamos el punto
# medio de cada rango, ponderado por el monto de cada tramo.
tr1_monto, tr1_mid = 86781, (0.0003 + 0.0575) / 2   # emisiones 2013-2023
tr2_monto, tr2_mid = 4500,  (0.0407 + 0.0483) / 2   # emision 2025
tasa_td = (tr1_monto * tr1_mid + tr2_monto * tr2_mid) / (tr1_monto + tr2_monto)

# === kd = tasa efectiva promedio ponderada por el monto de deuda del 10-Q ===
kd = (cp * tasa_cp + term_debt * tasa_td) / deuda_total

# === Tasa impositiva efectiva — 10-Q, 6 meses terminados 28-mar-2026 ===
utilidad_antes_imp = 86835
provision_imp      = 15160
T = provision_imp / utilidad_antes_imp

# === kd despues de impuestos ===
kd_di = kd * (1 - T)

print(f"Tasa efectiva term debt (ponderada): {tasa_td:.4%}")
print(f"kd (antes de impuestos):             {kd:.4%}")
print(f"Tasa impositiva efectiva T:          {T:.4%}")
print(f"kd despues de impuestos:             {kd_di:.4%}")

Tasa efectiva term debt (ponderada): 2.9669%
kd (antes de impuestos):             2.9957%
Tasa impositiva efectiva T:          17.4584%
kd despues de impuestos:             2.4727%


In [2]:
# Tabla 1 - Variables usadas para calcular la deuda (montos 10-Q, tasas 10-K)
df_deuda = pd.DataFrame([
    {"Componente de deuda": "Commercial paper",
     "Monto (US$ M)": f"{cp:,}",        "Peso": f"{cp/deuda_total:.1%}",
     "Tasa efectiva": "4.19 %"},
    {"Componente de deuda": "Term debt - porcion corriente",
     "Monto (US$ M)": f"{td_corr:,}",   "Peso": f"{td_corr/deuda_total:.1%}",
     "Tasa efectiva": f"{tasa_td:.2%}"},
    {"Componente de deuda": "Term debt - no corriente",
     "Monto (US$ M)": f"{td_nocorr:,}", "Peso": f"{td_nocorr/deuda_total:.1%}",
     "Tasa efectiva": f"{tasa_td:.2%}"},
    {"Componente de deuda": "Total deuda con intereses (D)",
     "Monto (US$ M)": f"{deuda_total:,}", "Peso": "100.0%",
     "Tasa efectiva": f"{kd:.2%}  (= kd)"},
])
df_deuda

,Componente de deuda,Monto (US$ M),Peso,Tasa efectiva
0,Commercial paper,"1,997",2.4%,4.19 %
1,Term debt - porcion corriente,"8,310",9.8%,2.97%
2,Term debt - no corriente,"74,404",87.8%,2.97%
3,Total deuda con intereses (D),"84,711",100.0%,3.00% (= kd)


In [3]:
# Tabla 2 - Resultados: kd, tasa de impuestos y kd despues de impuestos
df_resultados = pd.DataFrame([
    {"Concepto": "kd  -  costo de la deuda antes de impuestos",
     "Formula": "promedio ponderado de tasas efectivas", "Valor": f"{kd:.2%}"},
    {"Concepto": "T  -  tasa impositiva efectiva",
     "Formula": "Provision impuestos / Utilidad antes de impuestos", "Valor": f"{T:.2%}"},
    {"Concepto": "kd despues de impuestos",
     "Formula": "kd x (1 - T)", "Valor": f"{kd_di:.2%}"},
])
df_resultados

,Concepto,Formula,Valor
0,kd - costo de la deuda antes de impuestos,promedio ponderado de tasas efectivas,3.00%
1,T - tasa impositiva efectiva,Provision impuestos / Utilidad antes de impuestos,17.46%
2,kd despues de impuestos,kd x (1 - T),2.47%


### Interpretación · costo de la deuda de Apple

Con las tasas efectivas del 10-K aplicadas a la deuda del 10-Q, el **costo de la
deuda antes de impuestos es kd ≈ 3.00 %**. La deuda de Apple es casi toda *term
debt* (97.6 % del total), así que kd queda dominado por la tasa efectiva de esos
bonos (~2.97 %); el commercial paper (4.19 %) pesa poco por su tamaño.

Aplicando la **tasa impositiva efectiva de 17.46 %** (provisión de impuestos ÷
utilidad antes de impuestos del 10-Q), el **escudo fiscal** reduce el costo a
**kd después de impuestos ≈ 2.47 %**: cada dólar de interés le cuesta a Apple
~17 % menos porque es deducible.

**Nota sobre la tasa del term debt.** El 10-K no publica un promedio único, sino
**rangos** de tasa efectiva por tramo (0.03 %–5.75 % en las emisiones 2013–2023 y
4.07 %–4.83 % en la emisión 2025). Tomamos el punto medio de cada rango ponderado
por el monto del tramo (≈ 2.97 %). Ese nivel tan bajo refleja que la mayor parte
de la deuda de Apple son *Notes* antiguos emitidos con cupones muy bajos.

**Costo embebido vs. costo marginal.** El kd ≈ 3.0 % es el **costo histórico
(embebido)** de la deuda ya existente. El **costo marginal** —lo que Apple pagaría
por endeudarse *hoy*— es más alto: su commercial paper rinde 4.19 % y sus notas
nuevas de 2025 tienen tasas efectivas de 4.07 %–4.83 %, lo que apunta a un kd
marginal de ~4.4 % (≈ 3.7 % después de impuestos). Para un WACC prospectivo suele
preferirse el costo marginal; para reflejar la carga real actual de la deuda, el
embebido.

| Enfoque | kd (antes) | kd (después de impuestos) |
|---|---:|---:|
| Embebido (deuda existente) | 3.00 % | 2.47 % |
| Marginal (deuda nueva hoy) | ~4.4 % | ~3.7 % |

> **En contexto:** en cualquiera de los dos enfoques, la deuda es **mucho más
> barata que el capital propio** de Apple (kₑ ≈ 5.3 % por Gordon, y más alto por
> CAPM). El escudo fiscal la abarata todavía más. Estos bloques —kₑ, kd y los pesos
> de deuda y capital— son los insumos del WACC.